In [21]:
import pandas as pd
import numpy as np
import re
import tools

In [22]:
# Suma kontrolna
nazwa_pliku = "fact_inka_hard_flagged_wazone.parquet"
moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")

Mój hash (posortowane):   fact_inka_hard_flagged_wazone.parquet   7687cf50ac8d617b1975ae3f76bf4401560fefabc47d9ddbaf72f908c10e2612


In [23]:
df_wazone = pd.read_parquet(f"dane/interim/{nazwa_pliku}")

In [24]:
# ============================================================
# Kategoria ABC(D) — Pareto na wartości sprzedaży, per TowId
# ============================================================

# 1) Suma wartości sprzedaży per TowId (z kalendarza — puste dni mają Wartosc=0, więc nie zaburzają sumy)
wartosc_towid = (
    df_wazone.groupby('TowId')['Wartosc'].sum()
    .reset_index()
    .sort_values('Wartosc', ascending=False)
    .reset_index(drop=True)
)

wartosc_towid['WartoscSkumulowanaPct'] = (
    wartosc_towid['Wartosc'].cumsum() / wartosc_towid['Wartosc'].sum() * 100
)

# 2) Czysty podział Pareto — bez uwzględniania martwych
def przypisz_abc(pct):
    if pct <= 80:
        return 'A'
    elif pct <= 95:
        return 'B'
    else:
        return 'C'

wartosc_towid['KategoriaABC'] = wartosc_towid['WartoscSkumulowanaPct'].apply(przypisz_abc)

print("Czysty podział Pareto (bez D):")
print(wartosc_towid['KategoriaABC'].value_counts())

# 3) Praktyczny wariant ABCD — martwe nadpisują D (kandydaci do wycofania)
mapa_martwy = df_wazone.drop_duplicates('TowId').set_index('TowId')['JestMartwy']
wartosc_towid['JestMartwy'] = wartosc_towid['TowId'].map(mapa_martwy)

wartosc_towid['KategoriaABCD'] = wartosc_towid['KategoriaABC']
wartosc_towid.loc[wartosc_towid['JestMartwy'] == 1, 'KategoriaABCD'] = 'D'

print("\nPraktyczny podział ABCD (martwe → D):")
print(wartosc_towid['KategoriaABCD'].value_counts())

# 4) Nie gub sygnału — które D-ki BYŁY ważne (A lub B), zanim umarły
wartosc_towid['AlarmWazneMartwe'] = (
    (wartosc_towid['KategoriaABCD'] == 'D') &
    (wartosc_towid['KategoriaABC'].isin(['A', 'B']))
)

print(f"\nAlarm — dawne A/B, teraz martwe: {wartosc_towid['AlarmWazneMartwe'].sum()}")

# 5) Domapowanie do df_wazone, żeby mieć wszystkie flagi w jednym miejscu
for kol in ['KategoriaABC', 'KategoriaABCD', 'AlarmWazneMartwe']:
    mapa = wartosc_towid.set_index('TowId')[kol]
    df_wazone[kol] = df_wazone['TowId'].map(mapa)

Czysty podział Pareto (bez D):
KategoriaABC
C    6620
B    3084
A    2777
Name: count, dtype: int64

Praktyczny podział ABCD (martwe → D):
KategoriaABCD
C    3937
D    3120
A    2727
B    2697
Name: count, dtype: int64

Alarm — dawne A/B, teraz martwe: 437


In [25]:
df_wazone['AlarmWazneMartwe']=df_wazone['AlarmWazneMartwe'].astype('int8')  

In [26]:
nazwa_pliku = "fact_inka_hard_flagged_pareto.parquet"
df_wazone.to_parquet(f"dane/interim/{nazwa_pliku}")


In [27]:
# Suma kontrolna

moj_hash = tools.hash_danych_bezpieczny(f"dane/interim/{nazwa_pliku}")
print(f"Mój hash (posortowane):   {nazwa_pliku}   {moj_hash}")
# Suma kontrolna e4703058981acc9c505f3b520ef91352862101c1f048d1112e7ef346676a10b4

Mój hash (posortowane):   fact_inka_hard_flagged_pareto.parquet   e4703058981acc9c505f3b520ef91352862101c1f048d1112e7ef346676a10b4
